# Most Asked Pandas Scenario-Based Questions

Complete code solutions for all 24 scenarios commonly asked in data engineering / data science interviews.

In [1]:
import pandas as pd
import numpy as np

---
## 🔷 Data Cleaning Scenarios

### Q1. Find columns with >30% missing values and drop them

In [2]:
# Sample DataFrame with sparse columns
np.random.seed(42)
df = pd.DataFrame(np.random.randn(100, 5), columns=[f'col_{i}' for i in range(5)])

# Inject missing values: col_2 has 40%, col_4 has 35% nulls
df.loc[df.sample(frac=0.40).index, 'col_2'] = np.nan
df.loc[df.sample(frac=0.35).index, 'col_4'] = np.nan

print("Missing % before:")
print((df.isnull().mean() * 100).round(1))

# Identify and drop columns with > 30% missing
threshold = 0.30
cols_to_drop = df.columns[df.isnull().mean() > threshold]
df_clean = df.drop(columns=cols_to_drop)

print(f"\nDropped columns: {list(cols_to_drop)}")
print(f"Remaining columns: {list(df_clean.columns)}")

Missing % before:
col_0     0.0
col_1     0.0
col_2    40.0
col_3     0.0
col_4    35.0
dtype: float64

Dropped columns: ['col_2', 'col_4']
Remaining columns: ['col_0', 'col_1', 'col_3']


### Q2. Clean a column with mixed types like "25years", "30yrs", and integers

In [3]:
df = pd.DataFrame({'age': [25, '30yrs', '28years', '35 yrs', 40, None, 'forty']})

# Extract numeric part using regex; coerce non-numeric leftovers to NaN
df['age_clean'] = (
    df['age']
    .astype(str)
    .str.extract(r'(\d+)', expand=False)  # grab first run of digits
    .pipe(pd.to_numeric, errors='coerce')  # 'forty' -> NaN
)

print(df)

       age  age_clean
0       25       25.0
1    30yrs       30.0
2  28years       28.0
3   35 yrs       35.0
4       40       40.0
5     None        NaN
6    forty        NaN


### Q3. Convert salary column with values like "50,000", "$60,000", "70000" to numeric

In [4]:
df = pd.DataFrame({'salary': ['50,000', '$60,000', '70000', '$1,20,000', None]})

# Strip $, commas, then cast
df['salary_clean'] = (
    df['salary']
    .str.replace(r'[$,]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)

print(df)

      salary  salary_clean
0     50,000       50000.0
1    $60,000       60000.0
2      70000       70000.0
3  $1,20,000      120000.0
4        NaN           NaN


### Q4. Remove duplicates based on 2 specific columns only

In [5]:
df = pd.DataFrame({
    'emp_id':   [1, 1, 2, 3, 3],
    'name':     ['Alice', 'Alice', 'Bob', 'Charlie', 'Charlie'],
    'dept':     ['HR', 'HR', 'IT', 'Finance', 'Finance'],
    'login_ts': ['2024-01-01', '2024-01-02', '2024-01-01', '2024-01-01', '2024-01-01'],
})

# Keep only the first occurrence of each (emp_id, dept) pair
df_deduped = df.drop_duplicates(subset=['emp_id', 'dept'], keep='first')
print(df_deduped)

   emp_id     name     dept    login_ts
0       1    Alice       HR  2024-01-01
2       2      Bob       IT  2024-01-01
3       3  Charlie  Finance  2024-01-01


### Q5. Standardize a date column with multiple formats

In [6]:
df = pd.DataFrame({'date_str': [
    '01-Jan-2023',
    '2023/01/01',
    'January 1 2023',
    '01/01/2023',
    '2023-01-01',
]})

# dayfirst=False and format=mixed (pandas>=2.0) handle heterogeneous formats safely
df['date'] = pd.to_datetime(df['date_str'], dayfirst=False, format='mixed')

print(df)
print(df.dtypes)

         date_str       date
0     01-Jan-2023 2023-01-01
1      2023/01/01 2023-01-01
2  January 1 2023 2023-01-01
3      01/01/2023 2023-01-01
4      2023-01-01 2023-01-01
date_str               str
date        datetime64[us]
dtype: object


---
## 🔷 GroupBy & Aggregation Scenarios

### Q6. Top 3 highest-paid employees in each department

In [8]:
df = pd.DataFrame({
    'name':   ['Alice','Bob','Charlie','David','Eve','Frank','Grace','Heidi'],
    'dept':   ['HR','HR','HR','IT','IT','IT','Finance','Finance'],
    'salary': [70000, 60000, 55000, 90000, 85000, 80000, 75000, 72000],
})

# Sort descending then take first 3 per group
top3 = (
    df.sort_values('salary', ascending=False)
      .groupby('dept', group_keys=False)
      .head(3)
)
print(top3)
print("="*50)

# Alternative: rank-based (handles ties gracefully)
df['rank'] = df.groupby('dept')['salary'].rank(method='dense', ascending=False)
top3_rank = df[df['rank'] <= 3].drop(columns='rank')
print(top3_rank)

      name     dept  salary
3    David       IT   90000
4      Eve       IT   85000
5    Frank       IT   80000
6    Grace  Finance   75000
7    Heidi  Finance   72000
0    Alice       HR   70000
1      Bob       HR   60000
2  Charlie       HR   55000
      name     dept  salary
0    Alice       HR   70000
1      Bob       HR   60000
2  Charlie       HR   55000
3    David       IT   90000
4      Eve       IT   85000
5    Frank       IT   80000
6    Grace  Finance   75000
7    Heidi  Finance   72000


### Q7. Month with highest sales for each city

In [11]:
df = pd.DataFrame({
    'city':  ['Mumbai','Mumbai','Mumbai','Delhi','Delhi','Delhi'],
    'month': ['Jan','Feb','Mar','Jan','Feb','Mar'],
    'sales': [100, 150, 120, 200, 180, 220],
})
print(df.groupby('city')['sales'].idxmax())

best_month = (
    df.loc[df.groupby('city')['sales'].idxmax()]
      [['city', 'month', 'sales']]
      .reset_index(drop=True)
)
print(best_month)

city
Delhi     5
Mumbai    1
Name: sales, dtype: int64
     city month  sales
0   Delhi   Mar    220
1  Mumbai   Feb    150


### Q8. Departments where average salary > overall company average

In [12]:
df = pd.DataFrame({
    'dept':   ['HR','HR','IT','IT','Finance','Finance'],
    'salary': [60000, 65000, 90000, 95000, 70000, 72000],
})

company_avg = df['salary'].mean()
print(company_avg)

dept_avg = df.groupby('dept')['salary'].mean()
print(dept_avg)

above_avg_depts = dept_avg[dept_avg > company_avg]

print(f"Company average salary: {company_avg:,.0f}")
print("\nDepts above company average:")
print(above_avg_depts)

75333.33333333333
dept
Finance    71000.0
HR         62500.0
IT         92500.0
Name: salary, dtype: float64
Company average salary: 75,333

Depts above company average:
dept
IT    92500.0
Name: salary, dtype: float64


### Q9. Customers who made more than 5 transactions in a single day

In [ ]:
df = pd.DataFrame({
    'customer_id': [1,1,1,1,1,1,2,2,2,3,3],
    'txn_date':    pd.to_datetime(['2024-01-01']*6 + ['2024-01-01']*3 + ['2024-01-02']*2),
    'amount':      [10,20,30,40,50,60,100,200,300,15,25],
})

txn_counts = df.groupby(['customer_id', 'txn_date']).size().reset_index(name='txn_count')
heavy_txn = txn_counts[txn_counts['txn_count'] > 5]
print(heavy_txn)

### Q10. Percentage contribution of each product to total sales

In [ ]:
df = pd.DataFrame({
    'product': ['A','B','C','D'],
    'sales':   [200, 350, 150, 300],
})

df['pct_contribution'] = (df['sales'] / df['sales'].sum() * 100).round(2)
print(df)

# Within each category group
df2 = pd.DataFrame({
    'category': ['Electronics','Electronics','Clothing','Clothing'],
    'product':  ['Phone','Laptop','Shirt','Pants'],
    'sales':    [300, 500, 100, 150],
})
df2['pct_within_category'] = (
    df2['sales'] / df2.groupby('category')['sales'].transform('sum') * 100
).round(2)
print(df2)

---
## 🔷 Merging & Joining Scenarios

### Q11. Merge produces more rows than expected — diagnosing and fixing duplicate keys

In [ ]:
# Problem: dept_id is NOT unique in salary table -> many-to-many join fan-out
employees = pd.DataFrame({
    'emp_id':  [1, 2, 3],
    'dept_id': [10, 10, 20],
    'name':    ['Alice', 'Bob', 'Charlie'],
})
salaries = pd.DataFrame({
    'dept_id': [10, 10, 20],       # duplicate dept_id 10!
    'salary':  [60000, 65000, 80000],
})

bad_merge = employees.merge(salaries, on='dept_id')
print(f"employees: {len(employees)} rows, salaries: {len(salaries)} rows")
print(f"After merge: {len(bad_merge)} rows (fan-out!)")
print(bad_merge)

# Diagnosis: check if join key is unique in both tables
print("\ndept_id duplicates in salaries:", salaries['dept_id'].duplicated().sum())

# Fix option 1: aggregate salaries first so dept_id becomes unique
salary_agg = salaries.groupby('dept_id')['salary'].mean().reset_index()
clean_merge = employees.merge(salary_agg, on='dept_id')
print(f"\nAfter fixing: {len(clean_merge)} rows")
print(clean_merge)

# Fix option 2: validate='many_to_one' to raise an error early if salary has dupes
try:
    employees.merge(salaries, on='dept_id', validate='many_to_one')
except Exception as e:
    print(f"\nvalidate caught: {e}")

### Q12. Merge when the join key has different column names

In [ ]:
employees = pd.DataFrame({'emp_id': [1, 2, 3], 'name': ['Alice', 'Bob', 'Charlie']})
salaries  = pd.DataFrame({'employee_id': [1, 2, 3], 'salary': [70000, 80000, 90000]})

# Use left_on / right_on when key names differ
merged = employees.merge(salaries, left_on='emp_id', right_on='employee_id')

# Drop the redundant key column
merged = merged.drop(columns='employee_id')
print(merged)

### Q13. Combine 12 monthly sales DataFrames into one

In [ ]:
import calendar

# Simulate 12 monthly DataFrames
monthly_dfs = [
    pd.DataFrame({'month': [calendar.month_abbr[m]], 'sales': [np.random.randint(1000, 5000)]})
    for m in range(1, 13)
]

# Concatenate all at once (avoid appending in a loop — O(n^2) copies)
annual = pd.concat(monthly_dfs, ignore_index=True)
print(annual)

# If reading from files:
# import glob
# files = glob.glob('data/sales_*.csv')
# annual = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

### Q14. Handling salary_x and salary_y after a merge

In [ ]:
df1 = pd.DataFrame({'emp_id': [1, 2], 'salary': [60000, 70000], 'bonus': [5000, 6000]})
df2 = pd.DataFrame({'emp_id': [1, 2], 'salary': [62000, 71000], 'dept': ['HR', 'IT']})

# _x/_y appear when both DataFrames share a non-key column name
merged = df1.merge(df2, on='emp_id')
print("Default merge (shows _x/_y):", merged.columns.tolist())
print(merged)

# Option 1: rename with custom suffixes to make intent clear
merged2 = df1.merge(df2, on='emp_id', suffixes=('_prev', '_curr'))
print("\nWith clear suffixes:")
print(merged2)

# Option 2: keep only one and drop the other
merged3 = df1.merge(df2, on='emp_id').drop(columns='salary_x').rename(columns={'salary_y': 'salary'})
print("\nKeep only current salary:")
print(merged3)

---
## 🔷 Feature Engineering Scenarios

### Q15. Create meaningful age groups from an age column (1–100)

In [ ]:
df = pd.DataFrame({'age': [5, 17, 25, 35, 45, 60, 75, 90]})

bins   = [0, 12, 17, 25, 35, 50, 65, 100]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Middle Aged', 'Senior', 'Elderly']

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)
print(df)

### Q16. Extract weekend vs weekday flag from a datetime column

In [ ]:
df = pd.DataFrame({'txn_date': pd.to_datetime([
    '2024-01-01',  # Monday
    '2024-01-06',  # Saturday
    '2024-01-07',  # Sunday
    '2024-01-10',  # Wednesday
])})

# dayofweek: 0=Monday … 6=Sunday
df['day_name']  = df['txn_date'].dt.day_name()
df['is_weekend'] = df['txn_date'].dt.dayofweek >= 5
df['day_type']  = df['is_weekend'].map({True: 'Weekend', False: 'Weekday'})

print(df)

### Q17. Split a free-text column like "Mumbai, Maharashtra" into two columns

In [ ]:
df = pd.DataFrame({'location': [
    'Mumbai, Maharashtra',
    'Bangalore, Karnataka',
    'Chennai, Tamil Nadu',
    'Delhi',        # no comma edge case
]})

# expand=True returns a DataFrame directly; n=1 limits to 1 split
split = df['location'].str.split(',', n=1, expand=True)
df['city']  = split[0].str.strip()
df['state'] = split[1].str.strip()   # NaN where there's no comma

print(df)

### Q18. Handle a categorical column with 500 unique values without creating 500 columns

In [ ]:
# Simulate high-cardinality column
np.random.seed(0)
categories = [f'cat_{i}' for i in range(500)]
df = pd.DataFrame({'product': np.random.choice(categories, size=10000), 'sales': np.random.randint(1,100, 10000)})

# Option 1: Target Encoding — replace category with mean of target
target_enc = df.groupby('product')['sales'].mean().rename('product_target_enc')
df['product_target_enc'] = df['product'].map(target_enc)
print("Target encoding sample:")
print(df.head())

# Option 2: Frequency Encoding — replace with how often the category appears
freq_enc = df['product'].value_counts(normalize=True).rename('product_freq_enc')
df['product_freq_enc'] = df['product'].map(freq_enc)
print("\nFrequency encoding sample:")
print(df.head())

# Option 3: Keep top-N categories, collapse rest to 'Other'
top_n = 10
top_cats = df['product'].value_counts().nlargest(top_n).index
df['product_binned'] = df['product'].where(df['product'].isin(top_cats), other='Other')
print(f"\nUnique values after top-{top_n} binning: {df['product_binned'].nunique()}")

### Q19. Number of days between a customer's first and last purchase

In [ ]:
df = pd.DataFrame({
    'customer_id': [1, 1, 1, 2, 2, 3],
    'purchase_date': pd.to_datetime([
        '2024-01-01', '2024-03-15', '2024-06-20',
        '2024-02-10', '2024-05-05',
        '2024-01-25',
    ]),
})

customer_tenure = (
    df.groupby('customer_id')['purchase_date']
      .agg(first_purchase='min', last_purchase='max')
      .assign(days_between=lambda x: (x['last_purchase'] - x['first_purchase']).dt.days)
      .reset_index()
)
print(customer_tenure)

---
## 🔷 Performance & Real-World Scenarios

### Q20. Reading a 10GB CSV when RAM is only 8GB

In [ ]:
# Strategy 1: chunked processing — iterate and aggregate without loading all at once
# (Using a small in-memory demo; replace 'file.csv' with your actual path)

import io

# Create a small demo CSV in memory to illustrate the pattern
csv_data = """dept,salary
HR,60000
IT,90000
HR,65000
Finance,75000
IT,85000"""

chunk_results = []
for chunk in pd.read_csv(io.StringIO(csv_data), chunksize=2):
    # Do aggregation per chunk
    chunk_results.append(chunk.groupby('dept')['salary'].sum())

# Combine partial results
final = pd.concat(chunk_results).groupby(level=0).sum()
print("Dept salary totals (chunked):", final.to_dict())

# Strategy 2: reduce dtypes upfront to cut memory usage
# dtype_map = {'salary': 'int32', 'dept': 'category'}
# df = pd.read_csv('file.csv', dtype=dtype_map, usecols=['dept','salary'])

# Strategy 3: use DuckDB or Polars which operate out-of-core
# import duckdb
# result = duckdb.sql("SELECT dept, SUM(salary) FROM 'file.csv' GROUP BY dept").df()

print("\nApproaches summary:")
print("1. pd.read_csv(..., chunksize=N)     – process in chunks")
print("2. dtype downcasting + usecols       – reduce columns/memory per row")
print("3. DuckDB / Polars                   – true out-of-core execution")

### Q21. Apply a complex function to 1 million rows efficiently

In [ ]:
import time

df = pd.DataFrame({'a': np.random.randint(1, 100, 100_000), 'b': np.random.randint(1, 100, 100_000)})

def complex_fn(row):
    return row['a'] ** 2 + row['b'] ** 2

# Slow: .apply() row by row (Python loop internally)
t0 = time.time()
df['result_apply'] = df.apply(complex_fn, axis=1)
print(f"apply():         {time.time()-t0:.3f}s")

# Fast: vectorized NumPy operations (no Python loop)
t0 = time.time()
df['result_vec'] = df['a'] ** 2 + df['b'] ** 2
print(f"vectorized:      {time.time()-t0:.3f}s")

# If you truly need row-level logic: numpy.vectorize or numba @jit
from numpy import vectorize

@vectorize
def fn_vec(a, b):
    return a ** 2 + b ** 2

t0 = time.time()
df['result_np_vec'] = fn_vec(df['a'].values, df['b'].values)
print(f"np.vectorize:    {time.time()-t0:.3f}s")

print(df.head())

### Q22. Fill missing dates in a time series DataFrame

In [ ]:
df = pd.DataFrame({
    'date':  pd.to_datetime(['2024-01-01','2024-01-02','2024-01-05','2024-01-07']),
    'sales': [100, 150, 200, 180],
})

# Create a complete daily date range and reindex
full_range = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
df_complete = (
    df.set_index('date')
      .reindex(full_range)
      .rename_axis('date')
      .reset_index()
)

# Fill missing sales: forward-fill, backward-fill, interpolate, or 0
df_complete['sales_ffill']       = df_complete['sales'].ffill()
df_complete['sales_interpolate'] = df_complete['sales'].interpolate(method='linear')
df_complete['sales_zero']        = df_complete['sales'].fillna(0)

print(df_complete)

### Q23. Apply different aggregation functions to different columns in a single groupby

In [ ]:
df = pd.DataFrame({
    'dept':       ['HR','HR','IT','IT','Finance','Finance'],
    'salary':     [60000, 65000, 90000, 95000, 70000, 72000],
    'bonus':      [3000, 4000, 8000, 9000, 5000, 6000],
    'experience': [3, 5, 7, 9, 4, 6],
})

# Named aggregation — clear and readable
result = df.groupby('dept').agg(
    avg_salary   = ('salary', 'mean'),
    total_bonus  = ('bonus', 'sum'),
    max_exp      = ('experience', 'max'),
    headcount    = ('salary', 'count'),
).reset_index()

print(result)

### Q24. Explode a column where each cell contains a list

In [ ]:
df = pd.DataFrame({
    'order_id': [101, 102, 103],
    'items':    [[1, 2, 3], [4, 5], [6]],
    'customer': ['Alice', 'Bob', 'Charlie'],
})

print("Before explode:")
print(df)

# explode() creates one row per element; index is preserved (reset if needed)
df_exploded = df.explode('items', ignore_index=True)
df_exploded['items'] = df_exploded['items'].astype(int)

print("\nAfter explode:")
print(df_exploded)

# If lists are stored as strings '[1,2,3]', parse first:
import ast
df_str = pd.DataFrame({'order_id':[201,202], 'items':['[1,2]','[3,4,5]']})
df_str['items'] = df_str['items'].apply(ast.literal_eval)
df_str = df_str.explode('items', ignore_index=True)
print("\nFrom string lists:")
print(df_str)